# Lorenz Water Wheel — Evaluate & compare (Colab)

A **test run**: no training. Clones the repo (which carries the committed weights
under `weights/`) and scores each model on `data/test-dataset` with the **same**
metrics, so the mean `|ω|` RMSE numbers are directly comparable.

Needs both TensorFlow (LSTM) and torch (TCN) — both are preinstalled on Colab. A GPU
runtime is optional but faster.

## 1. Get the code + committed weights

In [ ]:
# Clone the repo. BRANCH selects what to pull: "main" for the merged code, or a feature
# branch name to test changes before they are merged. --depth 1 grabs only the latest
# snapshot (faster); rm -rf makes re-running safe (a plain re-clone would silently keep
# the old dir).
BRANCH = "main"
%cd /content
!rm -rf Lorenz-wheel-prediction-LSTM
!git clone --depth 1 -b {BRANCH} https://github.com/antmatrik/Lorenz-wheel-prediction-LSTM.git
%cd Lorenz-wheel-prediction-LSTM

## 2. Install dependencies (both frameworks)

In [ ]:
!pip install -q -r requirements/eval.txt

## 3. Which weights are committed?

In [ ]:
!ls -la weights/

## 4. Score each model from `weights/`

Whichever model has no committed weights is skipped. The TCN uses its native
3-channel rollout; pass `--physics` to score it through the LSTM-style rollout
instead (isolates the architecture from the rollout strategy). Metrics are computed
identically for both, and written per model to `outputs/eval_<model>.csv` (each row
is one test file; the last row is the mean aggregate).

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

def run(cmd):
    print('\n$', cmd, flush=True)
    get_ipython().system(cmd)

LSTM = '--model lstm --weights weights/lstm.weights.h5 --stats weights/lstm_stats.npz'
TCN  = '--model tcn --checkpoint weights/tcn_checkpoint.pt'

# Per-model --output so the two runs don't overwrite each other's metrics CSV.
if os.path.exists('weights/lstm.weights.h5'):
    run(f'python src/common/evaluate.py {LSTM} --output outputs/eval_lstm.csv')
else:
    print('skip LSTM: weights/lstm.weights.h5 not committed yet')

if os.path.exists('weights/tcn_checkpoint.pt'):
    run(f'python src/common/evaluate.py {TCN} --output outputs/eval_tcn.csv')
else:
    print('skip TCN: weights/tcn_checkpoint.pt not committed yet')

## 5. Skill vs horizon (both models)

One rollout to the largest horizon, reported at each cutoff, with naive baselines
(`base0`, `persist`). Compare `|w|RMSE` (magnitude) across models. Each table is also
saved to `outputs/sweep_<model>.txt` so it goes into the download bundle.

In [ ]:
import os

def run(cmd):
    print('\n$', cmd, flush=True)
    get_ipython().system(cmd)

LSTM = '--model lstm --weights weights/lstm.weights.h5 --stats weights/lstm_stats.npz'
TCN  = '--model tcn --checkpoint weights/tcn_checkpoint.pt'

H = '25,50,100,200,400,800,1800'
# tee -> keep the sweep tables on disk so they go into the bundle too.
if os.path.exists('weights/lstm.weights.h5'):
    run(f'python src/common/evaluate.py {LSTM} --horizons {H} | tee outputs/sweep_lstm.txt')
if os.path.exists('weights/tcn_checkpoint.pt'):
    run(f'python src/common/evaluate.py {TCN} --horizons {H} | tee outputs/sweep_tcn.txt')

## 6. Bundle all results + download one file

Render the per-file plots (horizons 100/200/500/1800, up to 10 files each), then pack
**everything** into a single `outputs/eval_bundle.zip` and download it:

- `eval_lstm.csv`, `eval_tcn.csv` — per-file metrics + aggregate for each model
- `sweep_lstm.txt`, `sweep_tcn.txt` — the skill-vs-horizon tables
- `eval_plots_<model>_h<steps>.png` — stacked **signed ω** actual-vs-predicted plots
- `eval_plots_<model>_h<steps>_abs.png` — same in **|ω| magnitude** (fair view: forgives bifurcation direction flips)

Download that one file and hand it over for analysis.

In [ ]:
import os, glob, zipfile
os.makedirs('outputs', exist_ok=True)

def run(cmd):
    print('\n$', cmd, flush=True)
    get_ipython().system(cmd)

LSTM = '--model lstm --weights weights/lstm.weights.h5 --stats weights/lstm_stats.npz'
TCN  = '--model tcn --checkpoint weights/tcn_checkpoint.pt'

# Render per-file plots for each model (horizons 100,200,500,1800; up to 10 files each).
if os.path.exists('weights/lstm.weights.h5'):
    run(f'python src/common/evaluate.py {LSTM} --plots')
if os.path.exists('weights/tcn_checkpoint.pt'):
    run(f'python src/common/evaluate.py {TCN} --plots')

# Bundle EVERYTHING into ONE downloadable zip: metrics CSVs, horizon sweeps, plot PNGs.
artifacts = sorted(
    glob.glob('outputs/eval_lstm.csv') + glob.glob('outputs/eval_tcn.csv')
    + glob.glob('outputs/sweep_*.txt') + glob.glob('outputs/eval_plots_*.png')
)
with zipfile.ZipFile('outputs/eval_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in artifacts:
        z.write(f, arcname=os.path.basename(f))

print(f'\nbundled {len(artifacts)} files -> outputs/eval_bundle.zip')
for f in artifacts:
    print('  ', os.path.basename(f))

from google.colab import files
files.download('outputs/eval_bundle.zip')